# AGENTS026 – Hours 4–5: vLLM + PydanticAI RCA Agent

This notebook wires the retrieval layer into an RCA agent. It assumes you already have: incident candidates from Hours 2–3, and a FAISS vector store plus metadata from Hours 3–4.

In this step we:
- configure notebook-side vLLM client settings,
- define `RCAResult`,
- implement retrieval/helper tools,
- create a `RCAAgent` class backed by PydanticAI,
- and test it on 1–2 sample incidents.


## Section 1 – vLLM Serving Reference

Start the model server from a **terminal**, not from the notebook. Example command:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
  --served-model-name Qwen3-30B-A3B \
  --api-key abc-123 \
  --port 8000 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.92 \
  --max-num-seqs 8 \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --trust-remote-code
```

Recommended starting settings for this notebook:

- `--max-model-len 8192`: enough room for current signals + retrieved incidents + runbook chunks.
- `--max-num-seqs 8`: conservative for stability.
- `--gpu-memory-utilization 0.92`: high enough to use the GPU well while leaving some safety margin.

If you already started vLLM with a different model or name, just update the `MODEL_NAME` variable below to match.


## Section 2 – Install / Import Dependencies

We use PydanticAI with an OpenAI-compatible provider pointing at the local vLLM server.


In [2]:
%pip install -q pydantic-ai-slim openai faiss-cpu pandas pyarrow

print('Installed / ensured pydantic-ai-slim, openai, faiss-cpu, pandas, pyarrow')


Note: you may need to restart the kernel to use updated packages.
Installed / ensured pydantic-ai-slim, openai, faiss-cpu, pandas, pyarrow


In [8]:
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any
import os
import json
import asyncio

import pandas as pd
import numpy as np
import faiss
from pydantic import BaseModel, Field

from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

print('Imports OK')


Imports OK


## Section 3 – Paths, Inputs, and Core Schemas

Load incident candidates, vector index metadata, and recreate the structured schemas used by the RCA workflow.


In [2]:
root = Path.cwd() / 'agents026'
data_dir = root / 'data'
incidents_dir = data_dir / 'incidents'
vector_dir = data_dir / 'vectorstore'

incident_candidates_path = incidents_dir / 'incident_candidates.json'
index_path = vector_dir / 'incident_runbook.index'
meta_path = vector_dir / 'incident_runbook_metadata.parquet'
metrics_path = data_dir / 'metrics.csv'
changes_path = data_dir / 'change_events.csv'

with open(incident_candidates_path, 'r', encoding='utf-8') as f:
    incident_candidates_raw = json.load(f)

metrics_df = pd.read_csv(metrics_path, parse_dates=['timestamp'])
changes_df = pd.read_csv(changes_path, parse_dates=['timestamp'])
corpus_df = pd.read_parquet(meta_path)
faiss_index = faiss.read_index(str(index_path))

class IncidentCandidate(BaseModel):
    incident_id: str
    start_time: datetime
    end_time: datetime
    services: List[str]
    anomaly_type: str
    metric_summary: Dict[str, float] = Field(default_factory=dict)
    log_samples: List[str] = Field(default_factory=list)
    k8s_event_samples: List[str] = Field(default_factory=list)
    change_refs: List[str] = Field(default_factory=list)

class RCAResult(BaseModel):
    incident_id: str
    root_cause_hypothesis: str
    impacted_components: List[str]
    probable_trigger: Optional[str] = None
    evidence: List[str] = Field(default_factory=list)
    confidence: float = Field(..., ge=0.0, le=1.0)

incident_candidates = [IncidentCandidate(**x) for x in incident_candidates_raw]
incident_map = {x.incident_id: x for x in incident_candidates}

len(incident_candidates), faiss_index.ntotal


(4, 10)

## Section 4 – Notebook-side vLLM Client Configuration

These values control the client side. The real context window is determined by the server’s `--max-model-len`, but we also keep prompts compact and set generation limits here.


In [3]:
BASE_URL = os.environ.get('BASE_URL', 'http://localhost:8000/v1')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'abc-123')
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen3-30B-A3B')

os.environ['BASE_URL'] = BASE_URL
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['MODEL_NAME'] = MODEL_NAME

MAX_OUTPUT_TOKENS = 512
TEMPERATURE = 0.1

print({'BASE_URL': BASE_URL, 'MODEL_NAME': MODEL_NAME, 'MAX_OUTPUT_TOKENS': MAX_OUTPUT_TOKENS, 'TEMPERATURE': TEMPERATURE})


{'BASE_URL': 'http://localhost:8000/v1', 'MODEL_NAME': 'Qwen3-30B-A3B', 'MAX_OUTPUT_TOKENS': 512, 'TEMPERATURE': 0.1}


In [4]:
provider = OpenAIProvider(base_url=BASE_URL, api_key=OPENAI_API_KEY)
chat_model = OpenAIChatModel(MODEL_NAME, provider=provider)
print('OpenAI-compatible model client initialized for:', MODEL_NAME)


OpenAI-compatible model client initialized for: Qwen3-30B-A3B


## Section 5 – Helper Functions for Retrieval

We define helpers for:
- locating an incident by ID,
- retrieving current signals,
- retrieving nearby changes,
- retrieving similar incidents/runbooks from FAISS.


In [5]:
def incident_to_query_text(incident: IncidentCandidate) -> str:
    metric_bits = ', '.join([f'{k}={v:.4f}' for k, v in incident.metric_summary.items()])
    logs = ' | '.join(incident.log_samples[:5])
    k8s = ' | '.join(incident.k8s_event_samples[:3])
    changes = ' | '.join(incident.change_refs[:3])
    return (
        f'incident_id={incident.incident_id}; services={','.join(incident.services)}; '
        f'anomaly_type={incident.anomaly_type}; metrics=[{metric_bits}]; '
        f'logs=[{logs}]; k8s_events=[{k8s}]; changes=[{changes}]'
    )


In [6]:
# Recreate the same embedding model used in Hours 3–4 for query encoding
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import torch

embed_device = 'cuda' if torch.cuda.is_available() else 'cpu'
EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=embed_device)

def embed_query_text(text: str) -> np.ndarray:
    vec = embed_model.encode([text], convert_to_numpy=True, normalize_embeddings=True)
    return vec.astype('float32')

print('Embedding model ready on', embed_device)


Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model ready on cpu


## Section 6 – Agent Dependencies + Tool Functions

PydanticAI tools operate against a dependency object. This keeps the retrieval/data access logic clean and reusable.


In [9]:
class RCAAgentDeps(BaseModel):
    incident_map: Dict[str, IncidentCandidate]
    metrics_df: Any
    changes_df: Any
    corpus_df: Any

deps = RCAAgentDeps(
    incident_map=incident_map,
    metrics_df=metrics_df,
    changes_df=changes_df,
    corpus_df=corpus_df,
)


In [10]:
def _get_current_signals_impl(incident_id: str) -> Dict[str, object]:
    inc = incident_map[incident_id]
    service = inc.services[0]
    start_time = pd.Timestamp(inc.start_time)
    end_time = pd.Timestamp(inc.end_time)
    mask = (metrics_df['service'] == service) & (metrics_df['timestamp'] >= start_time - pd.Timedelta(minutes=10)) & (metrics_df['timestamp'] <= end_time + pd.Timedelta(minutes=10))
    sdf = metrics_df.loc[mask].sort_values('timestamp')
    if sdf.empty:
        return {'incident_id': incident_id, 'service': service, 'summary': {}, 'points': []}
    summary = {
        'cpu_max': float(sdf['cpu_utilization'].max()),
        'cpu_avg': float(sdf['cpu_utilization'].mean()),
        'latency_p95_max': float(sdf['latency_p95_ms'].max()),
        'latency_p95_avg': float(sdf['latency_p95_ms'].mean()),
        'error_rate_max': float(sdf['error_rate'].max()),
        'rps_avg': float(sdf['rps'].mean()),
    }
    points = sdf.tail(15)[['timestamp', 'cpu_utilization', 'latency_p95_ms', 'error_rate', 'rps']].copy()
    points['timestamp'] = points['timestamp'].astype(str)
    return {
        'incident_id': incident_id,
        'service': service,
        'summary': summary,
        'points': points.to_dict(orient='records'),
        'log_samples': inc.log_samples[:8],
        'k8s_event_samples': inc.k8s_event_samples[:5],
    }

def _get_recent_changes_impl(incident_id: str) -> List[Dict[str, object]]:
    inc = incident_map[incident_id]
    service = inc.services[0]
    start_time = pd.Timestamp(inc.start_time)
    end_time = pd.Timestamp(inc.end_time)
    mask = (changes_df['service'] == service) & (changes_df['timestamp'] >= start_time - pd.Timedelta(minutes=30)) & (changes_df['timestamp'] <= end_time + pd.Timedelta(minutes=10))
    rows = changes_df.loc[mask].sort_values('timestamp')
    if rows.empty:
        return []
    out = rows[['timestamp', 'service', 'change_type', 'description', 'version']].copy()
    out['timestamp'] = out['timestamp'].astype(str)
    return out.to_dict(orient='records')

def _get_similar_incidents_impl(incident_id: str, top_k: int = 5) -> List[Dict[str, object]]:
    inc = incident_map[incident_id]
    q = embed_query_text(incident_to_query_text(inc))
    scores, ids = faiss_index.search(q, top_k + 3)
    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        row = corpus_df.iloc[int(idx)].to_dict()
        # remove direct self-hit if metadata contains same incident id
        if str(row.get('doc_id')) == incident_id:
            continue
        row['score'] = float(score)
        rows.append(row)
        if len(rows) >= top_k:
            break
    return rows


## Section 7 – Create the RCAAgent with Tools

We instruct the model to produce a concise, evidence-based root-cause hypothesis in the `RCAResult` schema.


In [11]:
SYSTEM_PROMPT = '''
You are an SRE root-cause-analysis agent.
Your job is to analyze one incident candidate using current telemetry, similar historical incidents, and recent change events.
Rules:
1. Be evidence-based and avoid overclaiming.
2. Prefer concrete operational triggers like deployment change, config change, pool exhaustion, dependency slowness, restart storms, or traffic spikes.
3. Return a structured RCAResult object only.
4. Keep evidence specific and short.
5. Confidence must be between 0 and 1.
'''

rca_agent = Agent(
    model=chat_model,
    deps_type=RCAAgentDeps,
    output_type=RCAResult,
    system_prompt=SYSTEM_PROMPT,
)

@rca_agent.tool
def get_current_signals(ctx: RunContext[RCAAgentDeps], incident_id: str) -> Dict[str, object]:
    return _get_current_signals_impl(incident_id)

@rca_agent.tool
def get_similar_incidents(ctx: RunContext[RCAAgentDeps], incident_id: str) -> List[Dict[str, object]]:
    return _get_similar_incidents_impl(incident_id)

@rca_agent.tool
def get_recent_changes(ctx: RunContext[RCAAgentDeps], incident_id: str) -> List[Dict[str, object]]:
    return _get_recent_changes_impl(incident_id)

print('RCAAgent ready')


RCAAgent ready


## Section 8 – RCAAgent Wrapper Class

Wrap the base agent in a small class so later notebooks/modules can call it more cleanly.


In [28]:
class RCAAgentWrapper:
    def __init__(self, agent: Agent, deps: RCAAgentDeps):
        self.agent = agent
        self.deps = deps

    def build_prompt(self, incident_id: str) -> str:
        inc = self.deps.incident_map[incident_id]
        return (
            f'Analyze incident {incident_id}. '
            f'Services: {inc.services}. '
            f'Anomaly type: {inc.anomaly_type}. '
            f'Use the available tools to inspect current signals, similar incidents, and recent changes before returning RCAResult.'
        )

    async def analyze_async(self, incident_id: str) -> RCAResult:
        prompt = self.build_prompt(incident_id)
        result = await self.agent.run(prompt, deps=self.deps)
        return result.output

    def analyze(self, incident_id: str) -> RCAResult:
        return asyncio.run(self.analyze_async(incident_id))

RCAAgent = RCAAgentWrapper(rca_agent, deps)
print('RCAAgent wrapper initialized')


RCAAgent wrapper initialized


## Section 9 – Tool Sanity Checks

Before testing the LLM end-to-end, verify that the notebook-side retrieval helpers return useful context.


In [13]:
sample_incident_id = incident_candidates[0].incident_id
print('Sample incident:', sample_incident_id)
print('\nCurrent signals:')
current_signals = _get_current_signals_impl(sample_incident_id)
print(json.dumps(current_signals, indent=2)[:2500])

print('\nRecent changes:')
recent_changes = _get_recent_changes_impl(sample_incident_id)
print(json.dumps(recent_changes, indent=2)[:2000])

print('\nSimilar incidents:')
similar_items = _get_similar_incidents_impl(sample_incident_id, top_k=5)
print(json.dumps(similar_items, indent=2)[:2500])


Sample incident: inc-001

Current signals:
{
  "incident_id": "inc-001",
  "service": "catalog-api",
  "summary": {
    "cpu_max": 33.97805209973438,
    "cpu_avg": 27.001498639944515,
    "latency_p95_max": 179.47066211838842,
    "latency_p95_avg": 162.74705329742775,
    "error_rate_max": 0.0032061994146254,
    "rps_avg": 43.206154488822776
  },
  "points": [
    {
      "timestamp": "2026-06-10 12:15:00",
      "cpu_utilization": 20.287472604372475,
      "latency_p95_ms": 177.3568532901446,
      "error_rate": 0.0,
      "rps": 51.94309179670459
    },
    {
      "timestamp": "2026-06-10 12:16:00",
      "cpu_utilization": 33.97805209973438,
      "latency_p95_ms": 164.5756593366515,
      "error_rate": 0.0001882475280572,
      "rps": 50.18020431987258
    },
    {
      "timestamp": "2026-06-10 12:17:00",
      "cpu_utilization": 24.865688939576945,
      "latency_p95_ms": 167.72278395482002,
      "error_rate": 0.0016186251338325,
      "rps": 43.47076764327872
    },
    {
 

## Section 10 – Test RCAAgent on 1–2 Sample Incidents

Run the RCA agent on one or two incidents and inspect the structured output.


In [14]:
test_ids = [x.incident_id for x in incident_candidates[:2]]
test_ids


['inc-001', 'inc-002']

In [30]:
results = []
for iid in test_ids:
    try:
        out = await RCAAgent.analyze_async(iid)
        results.append(out)
        print('\n===== RCA RESULT FOR', iid, '=====')
        print(out.model_dump_json(indent=2))
    except Exception as e:
        print('Failed for', iid, ':', repr(e))



===== RCA RESULT FOR inc-001 =====
{
  "incident_id": "inc-001",
  "root_cause_hypothesis": "The error rate increase in catalog-api is likely due to a recent deployment (v2.1.8) and feature flag activation (v2.2.2) around 12:13-12:16. These changes may have introduced inefficiencies or errors in request handling, leading to higher error rates.",
  "impacted_components": [
    "catalog-api"
  ],
  "probable_trigger": "deploy",
  "evidence": [
    "Similar incidents (inc-003, inc-002) involved deployment changes and feature flags.",
    "Recent changes show a deploy at 12:16 and feature flag at 12:13.",
    "Error rates spiked after these changes."
  ],
  "confidence": 0.95
}

===== RCA RESULT FOR inc-002 =====
{
  "incident_id": "inc-002",
  "root_cause_hypothesis": "The error rate increase in catalog-api is likely due to a recent deployment or feature flag change, which may have introduced inefficiencies or bugs. Historical incidents show similar patterns where deployments caused erro

## Section 11 – Optional Fallback: Direct Prompt Without Tool Calls

If your chosen model struggles with tool calling in the current vLLM setup, this fallback path still lets you test prompt + schema output by injecting tool results manually into the prompt. Keep it here as a backup for demo stability.


In [31]:
fallback_agent = Agent(
    model=chat_model,
    output_type=RCAResult,
    system_prompt=SYSTEM_PROMPT + '\nUse only the supplied incident context. Do not ask for tools.'
)

async def run_fallback_rca(incident_id: str) -> RCAResult:
    ctx_blob = {
        'incident': incident_map[incident_id].model_dump(),
        'current_signals': _get_current_signals_impl(incident_id),
        'recent_changes': _get_recent_changes_impl(incident_id),
        'similar_incidents': _get_similar_incidents_impl(incident_id, top_k=5),
    }
    prompt = 'Analyze this incident context and return RCAResult as structured output:\n' + json.dumps(ctx_blob, default=str)
    result = await fallback_agent.run(prompt)
    return result.output

# Uncomment for troubleshooting:
# fb = asyncio.run(run_fallback_rca(test_ids[0]))
# print(fb.json(indent=2))


## Section 12 – Save RCA Results

Persist successful outputs so the next notebook can use them directly for remediation planning and reporting.


In [32]:
rca_results_path = incidents_dir / 'rca_results.json'
serializable = [r.model_dump() for r in results]
with open(rca_results_path, 'w', encoding='utf-8') as f:
    json.dump(serializable, f, default=str, indent=2)

print('Saved RCA results to:', rca_results_path)
serializable[:2]


Saved RCA results to: /workspace/agents026/data/incidents/rca_results.json


[{'incident_id': 'inc-001',
  'root_cause_hypothesis': 'The error rate increase in catalog-api is likely due to a recent deployment (v2.1.8) and feature flag activation (v2.2.2) around 12:13-12:16. These changes may have introduced inefficiencies or errors in request handling, leading to higher error rates.',
  'impacted_components': ['catalog-api'],
  'probable_trigger': 'deploy',
  'evidence': ['Similar incidents (inc-003, inc-002) involved deployment changes and feature flags.',
   'Recent changes show a deploy at 12:16 and feature flag at 12:13.',
   'Error rates spiked after these changes.'],
  'confidence': 0.95},
 {'incident_id': 'inc-002',
  'root_cause_hypothesis': 'The error rate increase in catalog-api is likely due to a recent deployment or feature flag change, which may have introduced inefficiencies or bugs. Historical incidents show similar patterns where deployments caused error spikes, and the current error rate aligns with those scenarios.',
  'impacted_components': [